# MCP Demo — MAGIC v22 Orders & Complaints + Microsoft Learn

This notebook demonstrates how to connect an AI agent to **two MCP servers simultaneously** using a shared conversation session:

| MCP Server | URL | Auth |
|---|---|---|
| **Microsoft Learn** | `https://learn.microsoft.com/api/mcp` | None (public) |
| **MAGIC v22** | `http://localhost:9898/mcp` | API Key Bearer |

## Scenarios covered
1. **Azure Learn** — Ask a documentation question via the MS Learn MCP server
2. **Make an Order** — Place a new order for customer *Dileep Kumar*
3. **Get Customer Orders** — Retrieve all orders for Dileep Kumar
4. **Register a Complaint** — File a complaint against the newly created order
5. **Get Complaint Details** — Retrieve the complaint that was just registered
6. **Resolve Complaint** — Resolve the complaint via the Order Fulfillment team

> **Prerequisites**  
> - `magic-v22-mcp` server running at `http://localhost:9898/mcp`  
> - `az login` completed (for Azure OpenAI auth)  
> - Both `.env` files present (workspace root + `0-mcp-servers/magic-v22-mcp/.env`)

## 1. Imports

In [1]:
import os

import httpx
from dotenv import load_dotenv

from agent_framework import MCPStreamableHTTPTool
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


## 2. Configuration

Loads Azure credentials from the workspace root `.env` and the MAGIC v22 API key from the MCP server's own `.env`.

In [2]:
# Workspace root .env — Azure OpenAI credentials
load_dotenv(override=True)

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME")
api_key = os.getenv("API_KEY")

print("Project Endpoint :", project_endpoint)
print("Model            :", model)
print("MAGIC API Key    :", "✓ loaded" if api_key else "✗ not found — check 0-mcp-servers/magic-v22-mcp/.env")

Project Endpoint : https://ramkumar-ms-foundry.services.ai.azure.com/api/projects/ramkumar-msf-project
Model            : gpt-4o
MAGIC API Key    : ✓ loaded


## 3. MCP Tools Setup

- **MS Learn** — public, no auth required  
- **MAGIC v22** — protected by API Key; passed as a `Bearer` header via a custom `httpx.AsyncClient`

In [3]:
# ── Microsoft Learn MCP (public, no auth) ─────────────────────────────────────
ms_learn_tool = MCPStreamableHTTPTool(
    name="Microsoft Learn MCP Tool",
    url="https://learn.microsoft.com/api/mcp",
)

# ── MAGIC v22 MCP (API Key bearer auth) ───────────────────────────────────────
# MCPStreamableHTTPTool accepts a custom httpx.AsyncClient so we can inject
# the Authorization header for every outbound request to our protected server.
magic_v22_tool = MCPStreamableHTTPTool(
    name="MAGIC v22 Orders and Complaints Tool",
    url="http://localhost:9898/mcp",
    http_client=httpx.AsyncClient(
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=30.0,
    ),
)

print("✓ MS Learn MCP tool ready")
print("✓ MAGIC v22 MCP tool ready")

✓ MS Learn MCP tool ready
✓ MAGIC v22 MCP tool ready


## 4. Agent & Session Setup

A **single shared session** is created so the agent remembers context across all scenarios — e.g. the order ID created in Scenario 2 is automatically available when registering a complaint in Scenario 4.

In [4]:
credential = AzureCliCredential()

client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=model,
    credential=credential,
)

agent = client.as_agent(
    name="OrdersComplaintsAgent",
    instructions=(
        "You are a helpful assistant that manages customer orders and complaints "
        "using the MAGIC v22 MCP server. You can also look up Microsoft documentation "
        "using the Microsoft Learn MCP tool. "
        "When performing order or complaint operations, always confirm the action taken "
        "and include the relevant IDs (order_id, complaint_id) in your response so they "
        "can be referenced in follow-up requests."
    ),
    tools=[ms_learn_tool, magic_v22_tool],
)

# Single session shared across all scenario cells
session = agent.create_session()

print(f"✓ Agent '{agent.name}' ready")

✓ Agent 'OrdersComplaintsAgent' ready


---
## Scenario 1 — Microsoft Learn MCP

Ask a documentation question that is answered using the MS Learn MCP server.

In [5]:
query = "How do I create an Azure Blob Storage container using the Azure CLI?"

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

User: How do I create an Azure Blob Storage container using the Azure CLI?

Agent:
To create an Azure Blob Storage container using the Azure CLI, follow these steps:

### Prerequisites
1. Install the Azure CLI ([Installation Guide](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli)).
2. Sign in to your Azure account:
   ```bash
   az login
   ```
3. Ensure that you have a storage account created. You can create one by executing:
   ```bash
   az storage account create --name <storage-account> --resource-group <resource-group> --location <region>
   ```

### Creating a Container
Run the following command:
```bash
az storage container create \
    --name <container-name> \
    --account-name <storage-account> \
    --auth-mode login
```

### Key Notes
- Replace `<container-name>` with the name of your Blob container.
- Replace `<storage-account>` with your Azure Storage account name.
- The `--auth-mode login` parameter ensures authorization using Microsoft Entra (formerly Azu

---
## Scenario 2 — Make an Order

Place a new order for customer **Dileep Kumar** using the MAGIC v22 `make_order` tool.  
The agent will generate a product SKU, units, and order amount.

In [6]:
query = (
    "Please create a new order for customer 'Dileep Kumar'. "
    "Use product SKU 'LAPTOP-PRO-X1', 2 units at $1499.99 each. "
    "Add a remark: 'Urgent delivery requested'. "
    "Please confirm the order details including the order_id and order_number."
)

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

User: Please create a new order for customer 'Dileep Kumar'. Use product SKU 'LAPTOP-PRO-X1', 2 units at $1499.99 each. Add a remark: 'Urgent delivery requested'. Please confirm the order details including the order_id and order_number.

Agent:
The order for customer **Dileep Kumar** has been successfully created. Here are the details:

- **Order ID:** 13  
- **Order Number:** ORD10013  
- **Product SKU:** LAPTOP-PRO-X1  
- **Quantity:** 2  
- **Total Amount:** $2999.98  
- **Remarks:** Urgent delivery requested  
- **Order Status:** PENDING  

Let me know if you need further assistance!


---
## Scenario 3 — Get Orders of the Customer

Retrieve all orders placed by **Dileep Kumar** to confirm the order just created.

In [ ]:
query = "Show me all orders for customer 'Dileep Kumar'."

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

---
## Scenario 4 — Register a Complaint

File a complaint against the order created in Scenario 2.  
The agent uses the `order_id` already in the session context.

In [ ]:
query = (
    "I would like to register a complaint for the order we just created for Dileep Kumar. "
    "The complaint is: 'The laptop arrived with a cracked screen and the keyboard is unresponsive.' "
    "This is a HIGH priority issue. Register it on behalf of 'Dileep Kumar'. "
    "Please share the complaint_id once registered."
)

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

---
## Scenario 5 — Get Complaint Details

Fetch the full details of the complaint just registered to verify it was stored correctly.

In [ ]:
query = "Get the full details of the complaint we just registered."

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

---
## Scenario 6 — Resolve the Complaint

Resolve the complaint using the **Order Fulfillment** team.  
Valid resolver teams: `ORDER_FULFILLMENT`, `CUSTOMER_SUPPORT`, `LOGISTICS`, `BILLING`, `RETURNS_AND_REFUNDS`, `QUALITY_ASSURANCE`, `TECHNICAL_SUPPORT`

In [ ]:
query = (
    "Please resolve the complaint we just registered. "
    "It should be resolved by the 'ORDER_FULFILLMENT' team. "
    "Use this resolution remark: "
    "'A replacement unit has been dispatched via express delivery. "
    "Customer will receive it within 2 business days. Apologies for the inconvenience.' "
    "Confirm the updated complaint status after resolving."
)

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")